# Interactive simulation checks: anemia screening

Verifies the IFA effect on hemoglobin, anemia-status assignment, hemoglobin-screening
coverage (baseline vs the `anemia_screening_vv` scenario), and hemoglobin-test
sensitivity/specificity. Ported from the research portfolio VnV notebook
`model_18.3_interactive_simulation_anemia_screening`; updated to the current Engine
(`vivarium.engine`) API and converted to assert-based checks.

Note: the source's `ifa_deleted_hemoglobin.exposure` and `first_anc_hemoglobin.exposure`
pipelines no longer exist in the model. The IFA-effect and hemoglobin-truth checks were
re-expressed against the live `hemoglobin.exposure` pipeline (IFA applied) and the raw
`hemoglobin_exposure` state column (IFA not applied); the first-trimester-only `first_anc`
check was dropped. Revisit these against whatever pipeline the researchers intend for the
screened value.

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.artifact import Artifact
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [ ]:
!pip list | grep vivarium

In [ ]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
# hemoglobin.exposure is the (IFA-modified) pipeline; hemoglobin_exposure is the raw state
# column the modifier is applied on top of.
COLS = ["anc_attendance", "oral_iron_intervention", "hemoglobin_exposure",
        "hemoglobin_screening_coverage", "ferritin_screening_coverage",
        "tested_hemoglobin", "anemia_status_during_pregnancy", "hemoglobin.exposure"]

def build_sim(scenario=None):
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    sim.step()  # step past the pregnancy timestep
    return sim

def screening_frame(sim):
    return sim.get_population(COLS)

def anemia_status_from(hb):
    return np.where(hb <= 70, "severe",
           np.where(hb <= 100, "moderate",
           np.where(hb <= 110, "mild", "not_anemic")))

In [ ]:
# Baseline scenario
sim = build_sim()
df = screening_frame(sim)
df[["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
    "tested_hemoglobin", "anemia_status_during_pregnancy"]].head()

## IFA raises the hemoglobin exposure of the treated

In [ ]:
# The IFA effect modifies the `hemoglobin.exposure` pipeline relative to the raw
# `hemoglobin_exposure` state column: treated (IFA) simulants get a positive shift,
# untreated simulants get none.
ifa = df.oral_iron_intervention == "ifa"
hb_delta = df["hemoglobin.exposure"] - df["hemoglobin_exposure"]
assert hb_delta[ifa].mean() > 0, "IFA did not raise pipeline hemoglobin for the treated"
assert np.allclose(hb_delta[~ifa], 0, atol=1e-6), \
    "untreated (non-IFA) simulants had a hemoglobin shift"

## Anemia status is ordered by hemoglobin

In [ ]:
# anemia_status_during_pregnancy is assigned to screened simulants from hemoglobin
# thresholds (severe <=70, moderate <=100, mild <=110, else not_anemic). Assert the
# assigned categories are ordered by mean hemoglobin.exposure.
known = ["severe", "moderate", "mild", "not_anemic"]
assigned = df[df.anemia_status_during_pregnancy.isin(known)]
assert len(assigned) > 0, "no simulants have an assigned anemia status"
order = assigned.groupby("anemia_status_during_pregnancy")["hemoglobin.exposure"].mean()
assert order["severe"] < order["moderate"] < order["mild"] < order["not_anemic"], \
    f"anemia-status categories not ordered by hemoglobin: {order.to_dict()}"

## Screening coverage: baseline

In [ ]:
# No screening without ANC; partial hemoglobin screening at ANC; no ferritin screening at baseline.
cov = df.groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert cov.loc["none"] == 0, "hemoglobin screening occurred among no-ANC simulants at baseline"
attended = cov.drop("none")
assert (attended > 0).all() and (attended < 1).all(), \
    f"expected partial baseline hemoglobin screening at ANC, got {attended.to_dict()}"
assert (~df.ferritin_screening_coverage).all(), "ferritin screening should be off at baseline"

## Hemoglobin test sensitivity / specificity

In [ ]:
# Among screened simulants: sensitivity P(test low | truly low) ~= 0.85,
# specificity P(test adequate | truly adequate) ~= 0.80. Truth is taken from the
# hemoglobin.exposure pipeline (< 100 g/L); revisit if the test screens a different value.
tested = df[df.tested_hemoglobin != "not_tested"].copy()
tested["truth"] = np.where(tested["hemoglobin.exposure"] < 100, "low", "adequate")
sens = (tested.loc[tested.truth == "low", "tested_hemoglobin"] == "low").mean()
spec = (tested.loc[tested.truth == "adequate", "tested_hemoglobin"] == "adequate").mean()
assert np.isclose(sens, 0.85, atol=0.05), f"hemoglobin-test sensitivity {sens:.3f} far from 0.85"
assert np.isclose(spec, 0.80, atol=0.05), f"hemoglobin-test specificity {spec:.3f} far from 0.80"

## Screening coverage: `anemia_screening_vv` scale-up scenario

In [ ]:
# In the anemia-screening VnV scenario, every ANC attendee is screened for hemoglobin and
# no-ANC simulants are still never screened.
vv = build_sim(scenario="anemia_screening_vv")
vv_cov = vv.get_population(["anc_attendance", "hemoglobin_screening_coverage"]) \
    .groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert vv_cov.loc["none"] == 0, "hemoglobin screening among no-ANC simulants in anemia_screening_vv"
assert (vv_cov.drop("none") == 1).all(), \
    f"expected 100% hemoglobin screening at ANC in anemia_screening_vv, got {vv_cov.to_dict()}"